# Email spam classification on a GPU

This experiment uses a hashed bag-of-words representation and a linear PyTorch classifier. It is a faster GPU baseline on the full dataset, separate from the CPU comparison of preprocessing and classical models in `main.ipynb`. The feature hashing step runs on the CPU; model training and evaluation run on CUDA. A stratified validation set selects the epoch, and the test set is used only for the final report. The hashing vectorizer has a fixed mapping and does not learn from evaluation data.

Dataset: [Email Spam Classification Dataset](https://www.kaggle.com/datasets/purusinghvi/email-spam-classification-dataset). Set `SPAM_DATA_PATH` to `combined_data.csv` before running.

**Data:** [Email Spam Classification Dataset](https://www.kaggle.com/datasets/purusinghvi/email-spam-classification-dataset). Place `combined_data.csv` here or set `SPAM_DATA_PATH`.

In [1]:
import os, random, time
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
assert torch.cuda.is_available(), "This experiment requires a CUDA GPU."
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0), flush=True)
df = pd.read_csv(os.environ.get("SPAM_DATA_PATH", "combined_data.csv"), encoding="utf-8")
df = df.dropna(subset=["text", "label"]).drop_duplicates(subset="text").reset_index(drop=True)
assert set(df.label.unique()) == {0, 1}
train_idx, holdout_idx = train_test_split(np.arange(len(df)), test_size=0.30, random_state=seed, stratify=df.label)
val_idx, test_idx = train_test_split(holdout_idx, test_size=0.50, random_state=seed, stratify=df.label.iloc[holdout_idx])
print(f"Messages: {len(df)}; train: {len(train_idx)}; validation: {len(val_idx)}; test: {len(test_idx)}", flush=True)

GPU: Tesla T4


Messages: 83446; train: 58412; validation: 12517; test: 12517


In [2]:
# Hashing has no fitted vocabulary; transform each split without learning from it.
vectorizer = HashingVectorizer(n_features=2**18, ngram_range=(1, 2), alternate_sign=False, norm="l2", dtype=np.float32)
start = time.monotonic()
X_train = vectorizer.transform(df.text.iloc[train_idx])
X_val = vectorizer.transform(df.text.iloc[val_idx])
X_test = vectorizer.transform(df.text.iloc[test_idx])
y_train = df.label.iloc[train_idx].to_numpy(dtype=np.float32)
y_val = df.label.iloc[val_idx].to_numpy(dtype=np.float32)
y_test = df.label.iloc[test_idx].to_numpy(dtype=np.int64)
print(f"Features prepared in {time.monotonic()-start:.1f}s", flush=True)

Features prepared in 26.4s


In [3]:
def sparse_batch(matrix, indices):
    coo = matrix[indices].tocoo()
    coordinates = torch.from_numpy(np.stack([coo.row, coo.col]).astype(np.int64)).to(device)
    values = torch.from_numpy(coo.data).to(device)
    return torch.sparse_coo_tensor(coordinates, values, coo.shape, device=device).coalesce()

class SparseClassifier(torch.nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.weight = torch.nn.Parameter(torch.zeros(num_features, 1, device=device))
        self.bias = torch.nn.Parameter(torch.zeros(1, device=device))
    def forward(self, x):
        return (torch.sparse.mm(x, self.weight) + self.bias).squeeze(1)

def evaluate(model, X, y, batch_size=2048):
    model.eval()
    predictions, losses = [], []
    with torch.no_grad():
        for start in range(0, len(y), batch_size):
            idx = np.arange(start, min(start + batch_size, len(y)))
            logits = model(sparse_batch(X, idx))
            targets = torch.as_tensor(y[idx], dtype=torch.float32, device=device)
            losses.append((torch.nn.functional.binary_cross_entropy_with_logits(logits, targets).item(), len(idx)))
            predictions.append((logits >= 0).cpu().numpy())
    return np.concatenate(predictions).astype(int), sum(v*n for v,n in losses)/len(y)

model = SparseClassifier(X_train.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
rng = np.random.default_rng(seed)
best_loss, best_state = float("inf"), None
for epoch in range(1, 7):
    started = time.monotonic()
    model.train()
    for indices in np.array_split(rng.permutation(len(y_train)), int(np.ceil(len(y_train)/2048))):
        x = sparse_batch(X_train, indices)
        y = torch.as_tensor(y_train[indices], dtype=torch.float32, device=device)
        optimizer.zero_grad()
        loss = torch.nn.functional.binary_cross_entropy_with_logits(model(x), y)
        loss.backward()
        optimizer.step()
    pred, val_loss = evaluate(model, X_val, y_val)
    print(f"Epoch {epoch}: val loss {val_loss:.4f}; val accuracy {accuracy_score(y_val, pred):.4f}; {time.monotonic()-started:.1f}s", flush=True)
    if val_loss < best_loss:
        best_loss = val_loss
        best_state = {key: value.detach().clone() for key, value in model.state_dict().items()}
model.load_state_dict(best_state)

/tmp/ipykernel_18/244316567.py:5: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  return torch.sparse_coo_tensor(coordinates, values, coo.shape, device=device).coalesce()


Epoch 1: val loss 0.2216; val accuracy 0.9766; 1.3s


Epoch 2: val loss 0.1349; val accuracy 0.9809; 0.5s


Epoch 3: val loss 0.1039; val accuracy 0.9827; 0.5s


Epoch 4: val loss 0.0871; val accuracy 0.9839; 0.5s


Epoch 5: val loss 0.0763; val accuracy 0.9851; 0.5s


Epoch 6: val loss 0.0688; val accuracy 0.9862; 0.5s


<All keys matched successfully>

In [4]:
test_predictions, test_loss = evaluate(model, X_test, y_test)
print(f"Held-out test accuracy: {accuracy_score(y_test, test_predictions):.4f}")
print(f"Held-out test loss: {test_loss:.4f}")
print(classification_report(y_test, test_predictions, target_names=["ham", "spam"], zero_division=0))
print("Confusion matrix (rows: true; columns: predicted):")
print(confusion_matrix(y_test, test_predictions))

Held-out test accuracy: 0.9879
Held-out test loss: 0.0662
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99      5930
        spam       0.99      0.99      0.99      6587

    accuracy                           0.99     12517
   macro avg       0.99      0.99      0.99     12517
weighted avg       0.99      0.99      0.99     12517

Confusion matrix (rows: true; columns: predicted):
[[5845   85]
 [  67 6520]]
